# Stomp Pad Studio: YOLO board detector

Two parts:

**A. Stock model (run today).** Exports Ultralytics' COCO-trained `yolov8n-seg`, which already knows the class *snowboard*, as a small ONNX file the website runs in the browser.

**B. Fine-tune (run once you've collected photos).** Trains the same model on the boards shared from the website: photos of boards lying flat, with the users' corrected outlines. It then compares the fine-tuned model with the stock one and with the colour detector, and exports it.

Outputs `board-yolo.onnx` and `yolo.json`. Put both in `stomp-pad/model/` in the repo. Licence: Ultralytics YOLO is AGPL-3.0; the site's code is public, which satisfies it.

In [ ]:
!pip -q install ultralytics onnx onnxslim
import os, sys, json, glob, random, shutil
import numpy as np
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
from ultralytics import YOLO
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive, files
    drive.mount('/content/drive')
DATA_DIR = '/content/drive/MyDrive/Stomp Pad Dataset'   # made by data/collector.gs
IMGSZ = 320          # the website runs the model at this size
OUT = '/content/export'; os.makedirs(OUT, exist_ok=True)

## A. Export the stock COCO model
COCO class 31 is *snowboard*. Skateboard (36) and surfboard (37) are accepted too, because a board lying flat sometimes scores as one of them.

In [ ]:
def export(model, name, board_classes, version):
    path = model.export(format='onnx', imgsz=IMGSZ, opset=17, simplify=True, dynamic=False, nms=False)
    shutil.copy(path, os.path.join(OUT, name))
    cfg = {'file': name, 'imgsz': IMGSZ, 'boardClasses': board_classes, 'conf': 0.2, 'version': version}
    json.dump(cfg, open(os.path.join(OUT, 'yolo.json'), 'w'), indent=1)
    print('exported', name, f'{os.path.getsize(os.path.join(OUT, name)) / 1e6:.1f} MB', cfg)

stock = YOLO('yolov8n-seg.pt')
export(stock, 'board-yolo.onnx', [31, 36, 37], 'coco-yolov8n-seg')
if IN_COLAB:
    files.download(os.path.join(OUT, 'board-yolo.onnx')); files.download(os.path.join(OUT, 'yolo.json'))

In [ ]:
# Optional: try the stock model on a photo of your own (upload one in the Files panel)
TEST = '/content/test.jpg'
if os.path.exists(TEST):
    r = stock.predict(TEST, imgsz=IMGSZ, conf=0.1, verbose=False)[0]
    print({stock.names[int(c)]: round(float(s), 2) for c, s in zip(r.boxes.cls, r.boxes.conf)})
    plt.figure(figsize=(8, 5)); plt.imshow(r.plot()[..., ::-1]); plt.axis('off'); plt.show()

## B. Fine-tune on the collected boards
Converts each shared photo and its corrected outline into YOLO segmentation format (one class, *board*) and trains from the COCO weights.

In [ ]:
labels = sorted(glob.glob(os.path.join(DATA_DIR, 'labels', '*.json')))
print(len(labels), 'boards collected')
assert len(labels) >= 20, 'Collect at least ~20 boards (ideally 100+) before fine-tuning.'

DS = '/content/boards_yolo'
shutil.rmtree(DS, ignore_errors=True)
items = [json.load(open(p)) for p in labels]
random.Random(0).shuffle(items)
n_val = max(4, len(items) // 5)
split = {'val': items[:n_val], 'train': items[n_val:]}
for part, its in split.items():
    os.makedirs(f'{DS}/images/{part}', exist_ok=True); os.makedirs(f'{DS}/labels/{part}', exist_ok=True)
    for d in its:
        src = os.path.join(DATA_DIR, 'images', d['id'] + '.jpg')
        if not os.path.exists(src): continue
        shutil.copy(src, f'{DS}/images/{part}/{d["id"]}.jpg')
        poly = ' '.join(f'{min(max(x, 0), 1):.5f} {min(max(y, 0), 1):.5f}' for x, y in d['final']['outline'])
        open(f'{DS}/labels/{part}/{d["id"]}.txt', 'w').write(f'0 {poly}\n')
open(f'{DS}/data.yaml', 'w').write(f'path: {DS}\ntrain: images/train\nval: images/val\nnames:\n  0: board\n')
print({k: len(v) for k, v in split.items()})

In [ ]:
# Boards are photographed at any angle and in any light, so rotate and colour-jitter heavily
model = YOLO('yolov8n-seg.pt')
model.train(data=f'{DS}/data.yaml', imgsz=IMGSZ, epochs=120, batch=16, patience=30,
            degrees=180, flipud=0.5, fliplr=0.5, hsv_h=0.5, hsv_s=0.7, hsv_v=0.4, scale=0.3,
            mosaic=0.5, project='/content/runs', name='board', exist_ok=True, verbose=False)
best = YOLO('/content/runs/board/weights/best.pt')
m = best.val(data=f'{DS}/data.yaml', imgsz=IMGSZ, verbose=False)
print(f'mask mAP50 {m.seg.map50:.3f}   mask mAP50-95 {m.seg.map:.3f}')

## Compare: fine-tuned YOLO vs stock YOLO vs the colour detector (validation photos)

In [ ]:
def poly_mask(outline, w, h):
    im = Image.new('L', (w, h), 0); ImageDraw.Draw(im).polygon([(x * w, y * h) for x, y in outline], fill=1); return np.array(im, bool)
def iou(a, b): u = (a | b).sum(); return float((a & b).sum() / u) if u else 1.0
def yolo_mask(model, path, classes, w, h):
    r = model.predict(path, imgsz=IMGSZ, conf=0.1, verbose=False, retina_masks=True)[0]
    if r.masks is None: return np.zeros((h, w), bool)
    keep = [i for i, c in enumerate(r.boxes.cls.int().tolist()) if c in classes]
    if not keep: return np.zeros((h, w), bool)
    i = max(keep, key=lambda i: float(r.boxes.conf[i]))
    return np.array(Image.fromarray(r.masks.data[i].cpu().numpy().astype(np.uint8) * 255).resize((w, h))) > 127

rows = []
for d in split['val']:
    p = os.path.join(DATA_DIR, 'images', d['id'] + '.jpg')
    if not os.path.exists(p): continue
    w, h = Image.open(p).size; gt = poly_mask(d['final']['outline'], w, h)
    rows.append({'fine-tuned YOLO': iou(yolo_mask(best, p, [0], w, h), gt),
                 'stock YOLO': iou(yolo_mask(stock, p, [31, 36, 37], w, h), gt),
                 'colour detector': iou(poly_mask(d['auto']['outline'], w, h), gt) if d.get('auto') else np.nan})
names = list(rows[0])
means = [np.nanmean([r[n] for r in rows]) for n in names]
found = [np.nanmean([r[n] > 0.8 for r in rows]) * 100 for n in names]
for n, mu, f in zip(names, means, found): print(f'{n:16s} mean IoU {mu:.3f}   found (IoU>0.8) {f:.0f}%')
fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
ax[0].bar(names, means, color=['#0e1a2b', '#3d8bff', '#7a8699']); ax[0].set_ylim(0, 1); ax[0].set_title('Mean IoU with the corrected outline')
ax[1].bar(names, found, color=['#0e1a2b', '#3d8bff', '#7a8699']); ax[1].set_ylim(0, 100); ax[1].set_title('% of photos where the board was found')
plt.tight_layout(); plt.show()

In [ ]:
export(best, 'board-yolo.onnx', [0], 'board-yolov8n-seg-ft')
if IN_COLAB:
    files.download(os.path.join(OUT, 'board-yolo.onnx')); files.download(os.path.join(OUT, 'yolo.json'))

## Put it on the website
Copy `board-yolo.onnx` and `yolo.json` into `stomp-pad/model/` and push, or send them to Claude. The site loads the model automatically and shows **YOLO** when it found the board. If YOLO doesn't find one, the site falls back to the colour detector.